In [ ]:
import os
import sys
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"  # Disable MPS memory limits

print(f"Python: {sys.executable}")
if "bento-ml-macos" not in sys.executable:
    raise RuntimeError(
        "This notebook is running the wrong interpreter.\n"
        "In the kernel picker, choose **Python (bento-ml-macos)** "
        "(or Kernel → Select Kernel), then rerun from the top."
    )

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

# Set device for Apple Silicon
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using MPS device: {device}")
    print(f"MPS memory limit disabled: {os.environ.get('PYTORCH_MPS_HIGH_WATERMARK_RATIO')}")
else:
    device = torch.device("cpu")
    print(f"Using CPU device: {device}")


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")

if token:
    login(token=token)
    print("Successfully logged into Hugging Face")
else:
    print("No Hugging Face token found. Please set HUGGINGFACE_TOKEN in your .env file")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Llama-2 7B; request access on Hugging Face if this repo is gated
model_id = "meta-llama/Llama-2-7b-hf"

print(f"Using model: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
tokenizer.padding_side = "right"

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Tokenizer model max length: {tokenizer.model_max_length}")
print(f"Built-in chat_template: {bool(getattr(tokenizer, 'chat_template', None))}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_chat(messages, add_generation_prompt=False):
    """Llama-2 instruction format. Base Llama-2 checkpoints have no chat_template."""
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    user = next((m["content"] for m in messages if m["role"] == "user"), "")
    assistant = next((m["content"] for m in messages if m["role"] == "assistant"), None)
    body = f"<<SYS>>\n{system}\n<</SYS>>\n\n{user}" if system else user
    text = f"<s>[INST] {body.strip()} [/INST]"
    if assistant is not None:
        text += f" {assistant.strip()}</s>"
    elif add_generation_prompt:
        text += " "
    return text


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch
from torch.utils.data import Dataset
import json

# Load the ARC dataset
raw = load_dataset("allenai/ai2_arc", "ARC-Easy")["train"]
print(f"Loaded {len(raw)} examples from ARC-Easy dataset")

# Use a smaller subset for memory-constrained training on macOS
raw = raw.select(range(min(200, len(raw))))  # Reduced from 500 to 200
print(f"Using {len(raw)} examples for training")


In [ ]:
def format_arc_example(data):
    """Format ARC example into prompt and full_text for training"""
    question = data["question"]
    choices = data["choices"]["text"]
    answer = data.get("answerKey")

    # Format choices
    formatted_choices = "\n".join(f"{chr(ord('A')+i)}. {txt}" for i, txt in enumerate(choices))

    # Messages for prompt only (system + user)
    messages_prompt = [
        {"role": "system", "content": "You are a careful multiple-choice solver."},
        {"role": "user", "content": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer with a single letter (A-D) only."}
    ]

    # Messages with assistant answer
    messages_full = messages_prompt + [
        {"role": "assistant", "content": answer}
    ]

    prompt_text = render_chat(messages_prompt, add_generation_prompt=False)
    full_text = render_chat(messages_full, add_generation_prompt=False)

    return {"prompt": prompt_text, "full_text": full_text}

formatted_data = [format_arc_example(raw[i]) for i in range(len(raw))]
print(f"Formatted {len(formatted_data)} examples")

print("\nSample prompt:\n", formatted_data[0]["prompt"][:300], "...")
print("\nSample full text:\n", formatted_data[0]["full_text"][:300], "...")



In [ ]:
class ARCDataset(Dataset):
    """
    Expects each item to be one of:
      { "prompt": <string without assistant answer>,
        "full_text": <prompt + assistant answer> }
    or legacy:
      { "text": <prompt + assistant answer> }  # will try to infer prompt via delimiter
    """
    def __init__(self, data, tokenizer, max_length=1024, assistant_delim="Assistant:"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.assistant_delim = assistant_delim

    def __len__(self):
        return len(self.data)

    def _split_legacy_text(self, text):
        """Best-effort split if only 'text' is provided."""
        if self.assistant_delim in text:
            # use the last occurrence to guard against 'Assistant:' in question text
            cut = text.rfind(self.assistant_delim)
            prompt = text[:cut]
            full_text = text  # already prompt + answer
        else:
            # fallback: treat entire thing as 'full_text' and no prompt (won't mask)
            prompt = ""
            full_text = text
        return prompt, full_text

    def __getitem__(self, idx):
        item = self.data[idx]

        if "prompt" in item and "full_text" in item:
            prompt_text = item["prompt"]
            full_text   = item["full_text"]
        elif "text" in item:
            prompt_text, full_text = self._split_legacy_text(item["text"])
        else:
            raise ValueError("Item must contain either ('prompt' & 'full_text') or 'text'.")

        # 1) Encode PROMPT ONLY (no padding) to get true prompt length in tokens
        enc_prompt = self.tokenizer(
            prompt_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            padding=False,
        )

        # 2) Encode FULL TEXT (prompt + assistant answer) with padding for the model
        enc_full = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            padding="max_length",
        )

        input_ids = enc_full["input_ids"].squeeze(0)
        attn_mask = enc_full["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_len = enc_prompt["input_ids"].size(1) if enc_prompt["input_ids"].ndim == 2 else int(enc_prompt["input_ids"].shape[-1])

        # Mask out everything before the assistant's answer
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "labels": labels,
        }


# Create dataset
train_dataset = ARCDataset(formatted_data, tokenizer, max_length=1024)
print(f"Created training dataset with {len(train_dataset)} examples")


In [ ]:
# Load model for fine-tuning with memory optimization
print("Loading model with memory optimizations...")

# Clear any existing memory
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Load model with memory-efficient settings
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=token,
    dtype=torch.float32,  # Use float32 for better MPS compatibility
    device_map="auto" if not torch.backends.mps.is_available() else None,
    low_cpu_mem_usage=True,     # Reduce memory usage during loading
    attn_implementation="eager"  # Use eager attention for better MPS compatibility
)

# Move model to device with memory management
if torch.backends.mps.is_available():
    try:
        model = model.to(device)
        print(f"Model loaded on MPS device: {next(model.parameters()).device}")
    except RuntimeError as e:
        if "out of memory" in str(e):
            print("MPS out of memory, falling back to CPU")
            device = torch.device("cpu")
            model = model.to(device)
            print(f"Model loaded on CPU device: {next(model.parameters()).device}")
        else:
            raise e
else:
    model = model.to(device)
    print(f"Model loaded on device: {next(model.parameters()).device}")

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

from peft import LoraConfig, get_peft_model, TaskType

# Llama uses q/k/v/o_proj (and MLP gate/up/down). GPT-2 names are kept as a fallback.
lora_suffixes = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    "c_attn", "c_proj",
}
targets = {
    name.rsplit(".", 1)[-1]
    for name, _ in model.named_modules()
    if name.rsplit(".", 1)[-1] in lora_suffixes
}
if not targets:
    raise ValueError(
        "No LoRA target modules found. Expected Llama projections "
        "(q_proj, k_proj, v_proj, o_proj, ...)."
    )

print("LoRA targets detected:", sorted(targets))

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=sorted(targets),
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Set MPS memory management
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    per_device_train_batch_size=1,              # Small batch size for macOS
    gradient_accumulation_steps=4,              # Reduced from 8 to save memory
    warmup_steps=10,
    learning_rate=5e-5,
    logging_steps=1,
    save_steps=50,
    eval_strategy="no",                         # Changed from evaluation_strategy
    save_strategy="steps",
    load_best_model_at_end=False,
    report_to=None,                             # Disable wandb/tensorboard
    use_mps_device=torch.backends.mps.is_available(),
    dataloader_pin_memory=False,                # Disable pin memory for macOS
    remove_unused_columns=False,
    fp16=False,                                 # fp16 not supported on MPS
    bf16=False,                                 # bf16 not supported on MPS
    dataloader_num_workers=0,                   # Reduce memory usage
    max_grad_norm=1.0,                          # Gradient clipping to help with memory
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Memory cleanup before training
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("🚀 Starting training...")
print(f"Training on device: {device}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Dataset size: {len(train_dataset)} examples")

trainer.train()
print("Training completed!")


In [ ]:
# Test the fine-tuned model
def test_model(model, tokenizer, question, choices):
    formatted_choices = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(choices))
    messages = [
        {"role":"system","content":"You are a careful multiple-choice solver."},
        {"role":"user","content":f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer with a single letter (A-D) only."}
    ]
    prompt = render_chat(messages, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.backends.mps.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1,         # <- only one token
            do_sample=False,          # <- greedy
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract just the new token
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    letter = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    # Optional: sanitize to A-D
    letter = (letter[:1].upper() if letter else "")
    if letter not in ["A","B","C","D"]:
        letter = "A"  # or empty; up to you
    return letter


# Test with a sample question
if len(formatted_data) > 0:
    sample = raw[0]  # Get original sample
    print("Sample question:")
    print(f"Question: {sample['question']}")
    print(f"Choices: {sample['choices']['text']}")
    print(f"Correct answer: {sample['answerKey']}")
    
    # Test the model
    response = test_model(model, tokenizer, sample['question'], sample['choices']['text'])
    print(f"\nModel response:\n{response}")
    
    print("\n Note: This is a simplified test. For production use, implement proper evaluation metrics.")
